In [0]:
# Importando bibliotecas e funções
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import last, monotonically_increasing_id

In [0]:
# Chamando a tabela raw_pizzaria

df_bronze = spark.read.table('tabela_raw_pizzaria')



In [0]:
# Criando uma coluna com a data extraida do campo "value" 

df_bronze = df_bronze.withColumn("data_bruta",F.regexp_extract(F.col("value"), r"- R igor:\s*(\d{2}/\d{2})", 1))
display(df_bronze)

In [0]:
from pyspark.sql.functions import col, when

# Aplicação da window function que vai criar e prencher todas as linhas com um id autogerado para que o spark entenda que é uma sequencia.

df_bronze = df_bronze.withColumn("id_linha", monotonically_increasing_id())
esteira = Window.orderBy("id_linha")

# Transforma o texto vazio ("") em Nulo (None)
df_bronze = df_bronze.withColumn(
    "data_bruta",
    when(col("data_bruta") == "", None).otherwise(col("data_bruta"))
)

# Criando a coluna "data_Final" que vai ignorar os valores vazios e preencher com o ultimo valor não nulo de "data_bruta" utilizando a "esteira" como referencia.

df_bronze = df_bronze.withColumn(
    "data_final",
    last("data_bruta", ignorenulls=True).over(esteira)
)

df_bronze.display()




In [0]:
# Selecionando e mantendo apenas as colunas que interessam para a criação da tabela bronze 

df_bronze_limpo = df_bronze.filter(F.col("data_final").isNotNull() & F.col("data_bruta").isNull())

df_bronze_limpo = df_bronze_limpo.select(F.col("data_final").alias("data"), F.col("value").alias("descricao_valor"))



                                   


In [0]:
#Salvando a camada bronze em tabela delta
df_bronze_limpo.write.mode("overwrite").format("delta").saveAsTable("tabela_bronze_pizzaria")